# Check Georeferenced Footprints

Load the Stage 2 footprint polygons, preview them on a Folium map, and save the interactive map to `data/metadata/footprints_map.html`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import folium
from pyproj import Transformer

In [ ]:
def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "georeferenced" / "footprints.geojson").is_file():
            return candidate
    raise FileNotFoundError("Could not find data/georeferenced/footprints.geojson from the current working directory.")


repo_root = find_repo_root()
footprints_path = repo_root / "data" / "georeferenced" / "footprints.geojson"
map_output_path = repo_root / "data" / "metadata" / "footprints_map.html"

with footprints_path.open("r", encoding="utf-8") as handle:
    footprint_collection = json.load(handle)

features = footprint_collection["features"]
len(features)

In [ ]:
to_wgs84 = Transformer.from_crs("EPSG:32632", "EPSG:4326", always_xy=True)

all_lats: list[float] = []
all_lons: list[float] = []
bounds_latlon: list[tuple[float, float]] = []

for feature in features:
    ring = feature["geometry"]["coordinates"][0]
    for x, y in ring:
        lon, lat = to_wgs84.transform(x, y)
        all_lons.append(lon)
        all_lats.append(lat)

center_lat = sum(all_lats) / len(all_lats)
center_lon = sum(all_lons) / len(all_lons)

footprint_map = folium.Map(location=[center_lat, center_lon], zoom_start=20, tiles="OpenStreetMap")

for feature in features:
    ring = feature["geometry"]["coordinates"][0]
    latlon_ring = []
    for x, y in ring:
        lon, lat = to_wgs84.transform(x, y)
        latlon_ring.append([lat, lon])
        bounds_latlon.append((lat, lon))
    folium.Polygon(
        locations=latlon_ring,
        color="#1f77b4",
        weight=2,
        fill=True,
        fill_opacity=0.18,
        popup=feature["properties"].get("filename", "unknown"),
        tooltip=feature["properties"].get("filename", "unknown"),
    ).add_to(footprint_map)

south = min(lat for lat, _ in bounds_latlon)
north = max(lat for lat, _ in bounds_latlon)
west = min(lon for _, lon in bounds_latlon)
east = max(lon for _, lon in bounds_latlon)
footprint_map.fit_bounds([[south, west], [north, east]])

map_output_path.parent.mkdir(parents=True, exist_ok=True)
footprint_map.save(str(map_output_path))
map_output_path

In [ ]:
footprint_map